# Activation patching: the causal check

See `src/patching.py` and `docs/PATCHING.md`. This is the only notebook in
the repo whose output supports a causal claim.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.model import load_model
from src.patching import layer_sweep
from src.metrics import aggregate_patch_effects, causal_bottleneck_layer
from src.figures import layerwise_effect_curve, save_figure
import pandas as pd

model = load_model('gemma-2-2b')
sinhala = 'ප‍රාන්සයේ අගනුවර'
english = 'The capital of France is'
clean_tokens = model.to_tokens(sinhala)
corrupt_tokens = model.to_tokens(english)
_, clean_cache = model.run_with_cache(clean_tokens)
correct_id = model.to_single_token(' Paris')
incorrect_id = model.to_single_token(' London')
results = layer_sweep(model, clean_tokens, corrupt_tokens, clean_cache, -1, correct_id, incorrect_id)
df = pd.DataFrame([{'layer': r.layer, 'effect': r.effect} for r in results])
agg = aggregate_patch_effects(df.assign(effect=df['effect']))
print('bottleneck layer:', causal_bottleneck_layer(agg))